# GeneVariate — borrow Colab's GPU for a LOCAL run 🧬⚡

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SciSpectator/genevariate/blob/main/notebooks/genevariate_colab_gpu.ipynb)

**You run GeneVariate on your own machine, unmodified.** This notebook turns the Colab GPU into a
remote **Ollama** endpoint. Your local GeneVariate sends every LLM + embedding call here instead of
to your weak local GPU — because the code already reads its Ollama URL from the `OLLAMA_HOST`
environment variable (`config.py`). Nothing in the program changes; you just export one variable.

```
  your laptop                          Google Colab (this notebook)
  ┌────────────────────┐   HTTPS       ┌─────────────────────────────┐
  │ genevariate (GUI)  │  tunnel  ───▶ │ Ollama on the GPU           │
  │ OLLAMA_HOST=<url> ──┼──────────────▶│ gemma4:e2b + nomic-embed    │
  └────────────────────┘               └─────────────────────────────┘
```

### ⚙️ First: turn on the GPU
**Runtime → Change runtime type → Hardware accelerator: GPU** (T4 is enough; L4/A100 = more VRAM), Save.

> Keep this Colab tab open the whole time you use GeneVariate. The tunnel URL is new every session.

## 1 · Confirm the GPU

In [ ]:
!nvidia-smi || print('⚠️  No GPU — Runtime → Change runtime type → GPU, then rerun.')

## 2 · Install & start Ollama on the GPU
Bound to `0.0.0.0` with open origins so the tunnel can reach it, and tuned for throughput.

In [ ]:
import os, subprocess, time, requests

!curl -fsSL https://ollama.com/install.sh | sh

srv_env = {**os.environ,
           'OLLAMA_HOST': '0.0.0.0:11434',   # listen on all interfaces
           'OLLAMA_ORIGINS': '*',            # accept the tunnel's Host/Origin
           'OLLAMA_KEEP_ALIVE': '-1',        # keep the model resident
           'OLLAMA_FLASH_ATTENTION': '1',
           'OLLAMA_NUM_PARALLEL': '4'}
subprocess.Popen(['ollama', 'serve'],
                 stdout=open('/content/ollama.log', 'w'),
                 stderr=subprocess.STDOUT, env=srv_env)
for _ in range(60):
    try:
        if requests.get('http://127.0.0.1:11434/api/tags', timeout=2).ok:
            print('✅ Ollama is serving on the GPU'); break
    except Exception:
        time.sleep(1)
else:
    print('⚠️  Ollama did not start — see /content/ollama.log')

## 3 · Pull the models GeneVariate asks for (with tag fallback)
The code requests `gemma4:e2b` and `gemma4-e2b-text:latest`. Those aren't public Ollama tags
(`e2b` is Google's **Gemma 3n E2B**), so we pull a real model and **alias** it to those exact
names with `ollama cp`. Your local program then finds the model by the name it already uses —
no source edits.

In [ ]:
import subprocess
def sh(cmd):
    print('$', cmd); return subprocess.run(cmd, shell=True).returncode

WANT = ['gemma4:e2b', 'gemma4-e2b-text:latest']
base = None
if sh('ollama pull gemma4:e2b') == 0:
    base = 'gemma4:e2b'
else:
    for cand in ['gemma3n:e2b', 'gemma3:4b', 'gemma2:2b']:
        if sh(f'ollama pull {cand}') == 0:
            base = cand; break
assert base, 'Could not pull any Gemma model — check /content/ollama.log'
print('base model =', base)
for tag in WANT:
    if base != tag:
        sh(f'ollama cp {base} {tag}')
sh('ollama pull nomic-embed-text')   # embeddings (RAG / pseudo-cohorts)
print('\n=== models available on this GPU ==='); !ollama list

## 4 · Expose the GPU Ollama with a public tunnel (Cloudflare — no account)
Prints an `https://…trycloudflare.com` URL. Copy it — that's your `OLLAMA_HOST`.

In [ ]:
import subprocess, re, time, os
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared

subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate',
                  '--url', 'http://localhost:11434'],
                 stdout=open('/content/cf.log', 'w'), stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(45):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9.-]+\.trycloudflare\.com', open('/content/cf.log').read())
    if m:
        PUBLIC_URL = m.group(0); break
print('\n' + '='*60)
print(' OLLAMA_HOST for your laptop:\n')
print('   ' + (PUBLIC_URL or '(failed — see /content/cf.log)'))
print('='*60)

## 5 · Verify the tunnel end-to-end (before you leave Colab)
Hits the **public** URL exactly like your laptop will. A `200` + a word of text means you're good.
A `403` means the tunnel's Host header is being rejected — use the ngrok fallback in the next cell.

In [ ]:
import requests
r = requests.post(PUBLIC_URL + '/api/chat',
                  json={'model': 'gemma4:e2b',
                        'messages': [{'role': 'user', 'content': 'Reply with one word: ok'}],
                        'stream': False}, timeout=180)
print('HTTP', r.status_code)
print('reply:', r.json().get('message', {}).get('content', '')[:120] if r.ok else r.text[:300])
print('\n✅ Endpoint reachable — copy the URL above into OLLAMA_HOST on your laptop.'
      if r.ok else '⚠️  Not reachable — try the ngrok fallback cell below.')

## 6 · (Fallback) ngrok, if Cloudflare returned 403
ngrok can rewrite the Host header to `localhost`, which sidesteps Ollama's origin check.
Needs a free authtoken from https://dashboard.ngrok.com/get-started/your-authtoken — paste it below.

In [ ]:
NGROK_TOKEN = ''  # <-- paste your free ngrok authtoken here, then run this cell
if NGROK_TOKEN:
    !pip install -q pyngrok
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    ngrok.kill()
    tunnel = ngrok.connect(11434, 'http', host_header='localhost:11434')
    PUBLIC_URL = tunnel.public_url.replace('http://', 'https://')
    print('OLLAMA_HOST for your laptop:\n   ' + PUBLIC_URL)
else:
    print('Skipped — only needed if the Cloudflare cell 403d. Paste a token to use it.')

## 7 · On your LOCAL machine — point GeneVariate at the Colab GPU
Run these in your **laptop's** terminal (the program stays local; only the LLM runs on Colab).
Replace the URL with the one printed above.

```bash
cd ~/Desktop/genevariate
python3 -m venv venv && source venv/bin/activate
pip install -e ".[analysis]"          # local deps for the app + analysis (no local LLM needed)

# --- the one line that offloads the GPU work to Colab ---
export OLLAMA_HOST="https://XXXX.trycloudflare.com"   # paste from cell 4/6
export OLLAMA_NUM_PARALLEL=8   # stops the app from trying to restart a *local* Ollama

# Launch the GUI locally — extraction now runs on Colab's GPU
genevariate

# Or the headless extractor (also pass --ollama-url so its per-request default isn't localhost):
genevariate-llm-extract --gpl GPL570 --limit 5 \
    --ollama-url "$OLLAMA_HOST" --output ./extract_out
```

**How to know it worked:** the app log shows `gemma4:e2b preloaded into VRAM` and extraction runs
at GPU speed while your local `nvidia-smi` stays idle. On Colab, `nvidia-smi` shows the model resident.

### Notes
* **BioLORD Phase 1c semantic curator** (torch/sentence-transformers) is the one piece that runs
  *in-process* on your laptop, not through Ollama — it's a small embedding model and runs fine on
  CPU. Add `--no-semantic` to the CLI (or leave it; it'll use CPU) if your machine has no usable GPU.
* **Keep this Colab tab alive.** If the runtime recycles, rerun cells 2–4 and re-export the new URL.
* **Long generations:** GeneVariate uses unlimited output. If a Cloudflare tunnel drops mid-stream,
  switch to the ngrok fallback (cell 6), which is steadier for long streams.